# Classical LBM Baselines

This notebook is the visual entrypoint for the current classical CFD baseline. It is not the source of truth for solver behavior; the package code and tests are. Its job is to make the first D1Q3 and D2Q9 operations inspectable before any quantum or resource-estimation route is attempted.

Pedagogical sequence used throughout: paper anchor -> compact equation -> operator or matrix object -> observable plot -> falsifiable check.

Research role: the periodic D2Q9 Taylor-Green card in `docs/implementation_plan.md` is tied to `QRE2`, `QRE4`, `LBM14`, and `CAR7`. `LBM14` is used here only as collision-comparator context; this notebook remains classical.

## How To Read This Notebook

For each concept, look for four things:

1. The equation being checked.
2. The visual object that makes the update concrete.
3. The observable used as evidence.
4. The numerical check that can fail.

Tracked outputs are intentionally kept for selected pedagogical figures. Raw arrays and large tables are not displayed.

In [ ]:
# Corpus anchors: QRE2, QRE4, LBM14, CAR7. Notebook-only visual dependency: seaborn.
from pathlib import Path
import sys
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import seaborn as sns
from pq_cfd import SimulationConfig, d2q9_diagnostic_cases, run_d1q3, run_d2q9, run_d2q9_diagnostics
from pq_cfd.d1q3 import D1Q3_C, D1Q3_W, analytic_d1q3_density
from pq_cfd.d2q9 import D2Q9_C, D2Q9_W, analytic_taylor_green_velocity
from pq_cfd.d2q9_diagnostics import grid_spacings, max_mach_number, mean_kinetic_energy, normalized_l2_norm, periodic_divergence, periodic_vorticity, relative_scalar_error
from pq_cfd.metrics import relative_l2_error
sns.set_theme(context='notebook', style='whitegrid', palette='colorblind')
plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 130})

def speed(velocity):
    return np.sqrt(velocity[0] ** 2 + velocity[1] ** 2)

def centered_limits(field):
    limit = float(np.max(np.abs(field)))
    return (-limit, limit)

def show_scalar_field(ax, field, title, cmap='viridis', center=False):
    kwargs = {'origin': 'lower', 'cmap': cmap}
    if center:
        kwargs['vmin'], kwargs['vmax'] = centered_limits(field)
    image = ax.imshow(field.T, **kwargs)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    return image
print(f'project_root={project_root}')
print('visual policy: selected rendered outputs kept; no raw field dumps')

## D1Q3: Minimal Diffusion/Advection Storyboard

D1Q3 stores three distribution populations per lattice site with velocities `c_i in {-1, 0, 1}` and weights `w_i = {1/6, 2/3, 1/6}`.

The BGK step is:

$$
f_i^*(x,t)=f_i(x,t)-\frac{1}{\tau}\left[f_i(x,t)-f_i^{eq}(x,t)\right],\qquad
f_i(x+c_i,t+1)=f_i^*(x,t).
$$

The observable is the passive-scalar density wave. The check is whether it advects/decays against the analytic sinusoidal reference while conserving mass.

Anchors: classical benchmark gate for `QRE2`, `QRE4`, `CAR7`; collision-operator comparison context `LBM14`.

In [ ]:
# Corpus anchors: QRE2, QRE4, LBM14, CAR7. D1Q3 visual baseline and mass/error check.
d1_config = SimulationConfig(grid_shape=(128,), steps=100, tau=0.8, initial_condition='sinusoidal', sample_interval=20, amplitude=0.02, advection_velocity=0.03)
d1 = run_d1q3(d1_config)
expected_d1 = analytic_d1q3_density(d1_config, d1_config.grid_shape[0], d1_config.steps)
d1_residual = d1.density - expected_d1
_fig, _axes = plt.subplots(2, 2, figsize=(12, 7))
_ax = _axes[0, 0]
_ax.axhline(0, color='0.75', linewidth=1)
for c, w, color in zip(D1Q3_C, D1Q3_W, sns.color_palette('colorblind', 3)):
    if c == 0:
        _ax.scatter([0], [0], s=350 * w, color=color, label=f'c={c}, w={w:.3g}')
        _ax.text(0, 0.12, 'rest', ha='center')
    else:
        _ax.arrow(0, 0, c * 0.8, 0, width=0.015, head_width=0.08, length_includes_head=True, color=color)
        _ax.text(c * 0.9, 0.12, f'c={c}\nw={w:.3g}', ha='center')
_ax.set_xlim(-1.15, 1.15)
_ax.set_ylim(-0.25, 0.35)
_ax.set_yticks([])
_ax.set_title('D1Q3 stencil and weights (QRE2/LBM14 context)')
_ax.set_xlabel('one lattice timestep')
_ax = _axes[0, 1]
_sequence = ['populations\nf_i', 'BGK\ncollision', 'periodic\nstreaming', 'moment\nrho=sum f_i']
for _index, _label in enumerate(_sequence):
    _ax.text(_index, 0.5, _label, ha='center', va='center', bbox={'boxstyle': 'round,pad=0.35', 'fc': '#f7f7f7', 'ec': '0.35'})
    if _index < len(_sequence) - 1:
        _ax.annotate('', xy=(_index + 0.72, 0.5), xytext=(_index + 0.28, 0.5), arrowprops={'arrowstyle': '->', 'lw': 1.5})
_ax.set_xlim(-0.5, len(_sequence) - 0.5)
_ax.set_ylim(0, 1)
_ax.axis('off')
_ax.set_title('Operation sequence: collide -> stream -> observe')
_ax = _axes[1, 0]
x = np.arange(d1.density.size)
for step, density in zip(d1.history.steps, d1.history.density):
    _ax.plot(x, density, label=f't={step}')
_ax.plot(x, expected_d1, color='black', linestyle='--', linewidth=1.2, label='analytic final')
_ax.set_title('Density wave history: observable rho(x,t)')
_ax.set_xlabel('lattice site')
_ax.set_ylabel('density')
_ax.legend(ncol=3, fontsize=8)
_ax = _axes[1, 1]
_ax.plot(x, d1_residual, color=sns.color_palette('colorblind')[3])
_ax.axhline(0, color='0.35', linewidth=1)
_ax.set_title('Final residual: numerical - analytic')
_ax.set_xlabel('lattice site')
_ax.set_ylabel('density residual')
_fig.suptitle('D1Q3 passive scalar baseline (QRE2, QRE4, LBM14, CAR7)', y=1.02)
_fig.tight_layout()
print('D1Q3 falsifiable checks')
print(f"relative_l2_error_density={d1.metrics['relative_l2_error_density']:.3e}")
print(f"mass_drift_relative={d1.metrics['mass_drift_relative']:.3e}")
assert d1.metrics['mass_drift_relative'] < 1e-12
assert np.isfinite(d1.metrics['relative_l2_error_density'])

In [ ]:
# Corpus anchors: QRE2, QRE4, LBM14, CAR7. D1Q3 density heatmap plus final analytic check.
density_history = np.vstack(d1.history.density)
_fig, _axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={'width_ratios': [1.4, 1.0]})
sns.heatmap(density_history, ax=_axes[0], cmap='mako', cbar_kws={'label': 'density'}, xticklabels=16, yticklabels=[str(step) for step in d1.history.steps])
_axes[0].set_title('D1Q3 sampled density evolution')
_axes[0].set_xlabel('lattice site')
_axes[0].set_ylabel('sampled timestep')
_axes[1].plot(x, d1.density, label='LBM final')
_axes[1].plot(x, expected_d1, linestyle='--', label='analytic final')
_axes[1].fill_between(x, expected_d1, d1.density, alpha=0.25, color=sns.color_palette('colorblind')[3], label='residual')
_axes[1].set_title('Final-vs-analytic visual check')
_axes[1].set_xlabel('lattice site')
_axes[1].set_ylabel('density')
_axes[1].legend(fontsize=8)
_fig.tight_layout()

## D2Q9: BGK Taylor-Green Baseline

D2Q9 stores nine populations per lattice node: one rest population, four axis-aligned populations, and four diagonal populations. Macroscopic fields are moments of those populations:

$$
\rho=\sum_i f_i,\qquad \rho\mathbf{u}=\sum_i \mathbf{c}_i f_i.
$$

The Taylor-Green vortex is useful because the low-Mach velocity field has a known viscous decay:

$$
\mathbf{u}(t)=\mathbf{u}(0)\exp[-\nu(k_x^2+k_y^2)t],\qquad \nu=c_s^2(\tau-1/2).
$$

The named observables are velocity error, vorticity error, kinetic-energy decay, divergence RMS, Mach number, and mass drift. These are the observables future QCFD route notes must justify rather than assuming full-field readout.

Anchors: `QRE2`, `QRE4`, `LBM14`, `CAR7`.

In [ ]:
# Corpus anchors: QRE2, QRE4, LBM14, CAR7. D2Q9 stencil, weights, and operation object.
d2_config = SimulationConfig(grid_shape=(32, 32), steps=30, tau=0.8, initial_condition='taylor_green', sample_interval=10, amplitude=0.02)
d2 = run_d2q9(d2_config)
weight_grid = np.full((3, 3), np.nan)
for (cx, cy), weight in zip(D2Q9_C, D2Q9_W):
    weight_grid[1 - cy, cx + 1] = weight
_fig, _axes = plt.subplots(1, 3, figsize=(13, 4))
_ax = _axes[0]
for _index, ((cx, cy), weight) in enumerate(zip(D2Q9_C, D2Q9_W)):
    if cx == 0 and cy == 0:
        _ax.scatter([0], [0], s=800 * weight, color='black')
    else:
        _ax.arrow(0, 0, 0.8 * cx, 0.8 * cy, width=0.012, head_width=0.08, length_includes_head=True)
    _ax.text(1.05 * cx, 1.05 * cy, f'{_index}', ha='center', va='center', fontsize=9)
_ax.set_aspect('equal')
_ax.set_xlim(-1.35, 1.35)
_ax.set_ylim(-1.35, 1.35)
_ax.set_title('D2Q9 velocity set c_i')
_ax.set_xlabel('c_x')
_ax.set_ylabel('c_y')
_ax.grid(True, linewidth=0.5)
sns.heatmap(weight_grid, annot=True, fmt='.3g', cmap='crest', cbar=False, ax=_axes[1], square=True)
_axes[1].set_title('D2Q9 weight matrix w_i')
_axes[1].set_xlabel('c_x = -1,0,1')
_axes[1].set_ylabel('c_y = 1,0,-1')
_sequence = ['f_i field', 'equilibrium\nf_i^eq', 'BGK\ncollision', 'streaming\nper c_i', 'observables\nu, omega, E']
_ax = _axes[2]
for _index, _label in enumerate(_sequence):
    _ax.text(_index, 0.5, _label, ha='center', va='center', fontsize=9, bbox={'boxstyle': 'round,pad=0.28', 'fc': '#f7f7f7', 'ec': '0.35'})
    if _index < len(_sequence) - 1:
        _ax.annotate('', xy=(_index + 0.67, 0.5), xytext=(_index + 0.33, 0.5), arrowprops={'arrowstyle': '->', 'lw': 1.4})
_ax.set_xlim(-0.55, len(_sequence) - 0.45)
_ax.set_ylim(0, 1)
_ax.axis('off')
_ax.set_title('Operation sequence for the benchmark')
_fig.suptitle('D2Q9 operator picture before quantum route work (QRE2, QRE4, LBM14, CAR7)', y=1.03)
_fig.tight_layout()
print('D2Q9 basic checks')
print(f"relative_l2_error_velocity={d2.metrics['relative_l2_error_velocity']:.3e}")
print(f"mass_drift_relative={d2.metrics['mass_drift_relative']:.3e}")
assert d2.metrics['mass_drift_relative'] < 1e-12
assert np.isfinite(d2.metrics['relative_l2_error_velocity'])

In [ ]:
# Corpus anchors: QRE2, QRE4, LBM14, CAR7. Taylor-Green fields and selected-observable checks.
nx, ny = d2.config.grid_shape
spacing = grid_spacings((nx, ny))
expected_velocity = analytic_taylor_green_velocity(d2.config, nx, ny, d2.config.steps)
velocity_error = speed(d2.velocity - expected_velocity)
vorticity = periodic_vorticity(d2.velocity, spacing=spacing)
divergence = periodic_divergence(d2.velocity, spacing=spacing)
history_steps = np.array(d2.history.steps)
ke_history = np.array([mean_kinetic_energy(rho, vel) for rho, vel in zip(d2.history.density, d2.history.velocity)])
expected_ke_history = np.array([mean_kinetic_energy(np.full((nx, ny), d2.config.base_density), analytic_taylor_green_velocity(d2.config, nx, ny, int(step))) for step in history_steps])
mach_history = np.array([max_mach_number(vel) for vel in d2.history.velocity])
_fig, _axes = plt.subplots(2, 3, figsize=(13, 7))
images = [show_scalar_field(_axes[0, 0], speed(d2.velocity), 'Final speed |u|', cmap='viridis'), show_scalar_field(_axes[0, 1], vorticity, 'Final vorticity omega', cmap='coolwarm', center=True), show_scalar_field(_axes[0, 2], divergence, 'Divergence field div u', cmap='coolwarm', center=True), show_scalar_field(_axes[1, 0], velocity_error, 'Velocity error magnitude', cmap='magma')]
for image, _ax in zip(images, _axes.flat[:4]):
    _fig.colorbar(image, ax=_ax, fraction=0.046, pad=0.04)
_axes[1, 1].plot(history_steps, ke_history, marker='o', label='LBM')
_axes[1, 1].plot(history_steps, expected_ke_history, linestyle='--', marker='s', label='analytic decay')
_axes[1, 1].set_title('Kinetic-energy decay')
_axes[1, 1].set_xlabel('timestep')
_axes[1, 1].set_ylabel('mean kinetic energy')
_axes[1, 1].legend(fontsize=8)
_axes[1, 2].plot(history_steps, mach_history, marker='o')
_axes[1, 2].axhline(0.1, color='0.35', linestyle='--', label='low-Mach guard')
_axes[1, 2].set_title('Max Mach gate')
_axes[1, 2].set_xlabel('timestep')
_axes[1, 2].set_ylabel('max Mach')
_axes[1, 2].legend(fontsize=8)
_fig.suptitle('Taylor-Green selected observables (QRE2, QRE4, LBM14, CAR7)', y=1.02)
_fig.tight_layout()
print('D2Q9 falsifiable observable checks')
print(f'vorticity_rms={normalized_l2_norm(vorticity):.3e}')
print(f'divergence_rms={normalized_l2_norm(divergence):.3e}')
print(f'max_mach={max_mach_number(d2.velocity):.3e}')
assert normalized_l2_norm(divergence) < 0.01
assert max_mach_number(d2.velocity) < 0.1

## Diagnostics As A Quantum Follow-Up Gate

A route should not move to quantum operators simply because one visual run looks plausible. The diagnostics API runs controlled Taylor-Green cases, attaches adjacent-grid observed orders, and records blockers.

For QCFD, this matters because every later route note must say which observable is estimated, which classical baseline it is compared against, and why full-field readout is not silently assumed.

Anchors: `QRE2`, `QRE4`, `LBM14`, `CAR7`, with readout discipline inherited from the project corpus.

In [ ]:
# Corpus anchors: QRE2, QRE4, LBM14, CAR7. Diagnostic gate visualization.
diagnostic_cases = d2q9_diagnostic_cases(grids=((16, 16), (32, 32)), tau_values=(0.8,), amplitude=0.02)
diagnostic_records = run_d2q9_diagnostics(diagnostic_cases)
records = sorted(diagnostic_records, key=lambda record: record.grid_spacing, reverse=True)
spacings = np.array([record.grid_spacing for record in records])
velocity_errors = np.array([record.velocity_relative_l2_error for record in records])
vorticity_errors = np.array([record.vorticity_relative_l2_error for record in records])
energy_errors = np.array([record.kinetic_energy_relative_error for record in records])
status_values = np.array([[1 if record.passed_for_quantum_followup else 0 for record in records]])
status_labels = [f'{record.grid_shape[0]}x{record.grid_shape[1]}' for record in records]
finest = records[-1]
observable_names = ['vel L2', 'vort L2', 'KE err', 'div RMS', 'Mach', 'mass drift']
observable_values = [finest.velocity_relative_l2_error, finest.vorticity_relative_l2_error, finest.kinetic_energy_relative_error, finest.divergence_rms, finest.max_mach, max(finest.mass_drift_relative, 1e-18)]
_fig, _axes = plt.subplots(2, 2, figsize=(12, 7), gridspec_kw={'height_ratios': [1.2, 0.8]})
_ax = _axes[0, 0]
_ax.loglog(spacings, velocity_errors, marker='o', label='velocity')
_ax.loglog(spacings, vorticity_errors, marker='s', label='vorticity')
_ax.loglog(spacings, energy_errors, marker='^', label='kinetic energy')
_ax.invert_xaxis()
_ax.set_title('Refinement errors: controlled Taylor-Green decay')
_ax.set_xlabel('grid spacing h')
_ax.set_ylabel('relative error')
_ax.legend(fontsize=8)
_ax = _axes[0, 1]
orders = [record.velocity_observed_order for record in records]
order_x = [_label for _label, order in zip(status_labels, orders) if order is not None]
order_y = [order for order in orders if order is not None]
_ax.bar(order_x, order_y, color=sns.color_palette('colorblind')[2])
_ax.axhline(1.5, color='0.35', linestyle='--', label='gate minimum')
_ax.set_title('Observed velocity order gate')
_ax.set_xlabel('fine grid')
_ax.set_ylabel('observed order')
_ax.legend(fontsize=8)
sns.heatmap(status_values, ax=_axes[1, 0], cmap=ListedColormap(['#d95f02', '#1b9e77']), vmin=0, vmax=1, cbar=False, annot=np.array([['pass' if value else 'hold' for value in status_values[0]]]), fmt='', xticklabels=status_labels, yticklabels=['quantum follow-up'])
_axes[1, 0].set_title('Diagnostic status strip')
_axes[1, 0].set_xlabel('grid')
_axes[1, 0].set_ylabel('')
_ax = _axes[1, 1]
_ax.bar(observable_names, observable_values, color=sns.color_palette('colorblind', len(observable_names)))
_ax.set_yscale('log')
_ax.set_title(f'Finest-grid observable dashboard ({finest.grid_shape[0]}x{finest.grid_shape[1]})')
_ax.set_ylabel('value (log scale)')
_ax.tick_params(axis='x', rotation=25)
_fig.suptitle('D2Q9 diagnostic gate before QCFD route work (QRE2, QRE4, LBM14, CAR7)', y=1.02)
_fig.tight_layout()
for record in records:
    print(f"grid={record.grid_shape}, velocity_error={record.velocity_relative_l2_error:.3e}, velocity_order={record.velocity_observed_order}, status={record.validation_status}, blocker={record.validation_blocker or 'none'}")
assert all((np.isfinite(value) for value in observable_values))
assert any((record.velocity_observed_order is not None for record in records))

## Next Research Use

The next project step is not to install a quantum package. It is to write route notes against the D2Q9 benchmark card in `docs/implementation_plan.md`.

A route note must state the paper IDs, operator/update form, encoding, loading/reloading cost, readout/sample strategy, and resource quantities before code is added. The visual checks above define the classical observable surface that route notes must preserve.